In [1]:
from rdkit.Chem import AllChem 
from numpy import zeros
from rdkit import DataStructs
from rdkit.Chem import MolFromSmiles
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from pandas import DataFrame, concat
from pickle import load
import matplotlib.pyplot as plt

In [2]:
def calc_morgan(mols):
    """ генерация молекулярных отпечатков по методу Моргана с радиусом 2 и длиной 2048
    """
    mfp_gen = AllChem.GetMorganGenerator(radius=2, ) 
    for_df = []
    for m in mols:
        arr = zeros((1,), dtype=int)
        DataStructs.ConvertToNumpyArray(mfp_gen.GetFingerprint(m), arr)
        for_df.append(arr)
    return DataFrame(for_df)

with open('../data/classifier/no_dubl_potency.pickle', 'rb') as file:
    potency = load(file)

In [3]:
molecules = [
    MolFromSmiles(mol) for mol in potency['SMILES']
]

In [4]:
X = calc_morgan(molecules)
Y = potency['Potency']
XY = concat((X, Y), axis=1)
XY

,0,1,2,3,4,5,6,7,8,9,...,2039,2040,2041,2042,2043,2044,2045,2046,2047,Potency
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,38.860425
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,32.587425
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,12.688650
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,26.926167
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,27.397933
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8249,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,35.481300
8250,0,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,39.810700
8251,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,50.118700
8252,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,31.622800


In [5]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20) # попробовать 20, 15

In [6]:
scaler_y = MinMaxScaler().fit(y_train.to_numpy().reshape(-1, 1))
y_train_scal = scaler_y.transform(y_train.to_numpy().reshape(-1, 1))
y_test_scal = scaler_y.transform(y_test.to_numpy().reshape(-1, 1))

In [ ]:
rf = RandomForestRegressor()
pg = {'n_estimators': [50, 100, 200, 300, 400, 500],
      'max_depth': [None, 1, 3, 5, 7, 10],
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(rf, pg, verbose=3, cv=cv)

gs.fit(x_train, y_train_scal.ravel())

Fitting 25 folds for each of 36 candidates, totalling 900 fits
[CV 1/25] END ..max_depth=None, n_estimators=50;, score=0.163 total time=  48.4s
[CV 2/25] END ..max_depth=None, n_estimators=50;, score=0.224 total time=  48.4s
[CV 3/25] END ..max_depth=None, n_estimators=50;, score=0.220 total time=  48.8s
[CV 4/25] END ..max_depth=None, n_estimators=50;, score=0.225 total time=  50.3s
[CV 5/25] END ..max_depth=None, n_estimators=50;, score=0.252 total time=  53.5s
[CV 6/25] END ..max_depth=None, n_estimators=50;, score=0.155 total time=  48.9s
[CV 7/25] END ..max_depth=None, n_estimators=50;, score=0.215 total time=  48.0s
[CV 8/25] END ..max_depth=None, n_estimators=50;, score=0.244 total time=  49.6s
[CV 9/25] END ..max_depth=None, n_estimators=50;, score=0.239 total time=  48.6s
[CV 10/25] END .max_depth=None, n_estimators=50;, score=0.229 total time=  45.4s
[CV 11/25] END .max_depth=None, n_estimators=50;, score=0.205 total time=  49.1s
[CV 12/25] END .max_depth=None, n_estimators=5

In [ ]:
gs.best_estimator_

In [ ]:
y_pred = gs.predict(x_test_scal)
print(f'R^2 = {r2_score(y_test_scal, y_pred)}')
print(f'RMSE = {root_mean_squared_error(y_test_scal, y_pred)}')